In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

from ksw import Cosmology, utils
from ksw import estimator
import camb
import healpy as hp

from ksw import Afunctionals

In [2]:
# Setup CAMB parameters
pars = camb.CAMBparams()
pars.WantTensors=True
pars.set_cosmology(H0=67.66, ombh2=0.02242, omch2=0.11933)
pars.InitPower.set_params(As=2.1056e-9, ns=0.9665, r=0.001)
cosmo = Cosmology(pars, verbose=False)

# Compute transfer functions
print("Computing transfer functions...")
cosmo.compute_transfer(lmax=300)

# Compute angular power spectra
print("Computing angular power spectra...")
#cosmo.compute_c_ell()

Computing transfer functions...
Computing angular power spectra...


In [3]:
from ksw.shape import Shape
# radii = np.logspace(0, 3, 10)  # Small number of radii for quick testing
radii = np.asarray([10000, 10100])

prim_shape = Shape.prim_local(ns=0.9665)
#radii = np.logspace(0, 3, 10)

cosmo.compute_transfer(lmax=300)
cosmo.add_prim_reduced_bispectrum_scalar_dL(prim_shape, radii)

cosmo.compute_transfer_tensor(lmax=300)
cosmo.add_prim_reduced_bispectrum_tensor_dL(prim_shape, radii)

Updated CAMB param: WantTensors from False to True.
Updated CAMB param: WantTensors from True to False.


In [4]:
# Create a estimator instance
red_bispectra = cosmo.red_bispectra
icov = lambda alm: alm
pol = ('T', 'E', 'B')
estimator_con = estimator.KSW(red_bispectra, icov, lmax=100, pol=pol, precision='double')

In [5]:
import healpy as hp
lmax = estimator_con.lmax
nelem = hp.Alm.getsize(lmax+1)
print(nelem)

5253


In [6]:
hp.synalm?


Signature: hp.synalm(cls, lmax=None, mmax=None, new=False, verbose=True)
Docstring:
Generate a set of alm given cl.
The cl are given as a float array. Corresponding alm are generated.
If lmax is None, it is assumed lmax=cl.size-1
If mmax is None, it is assumed mmax=lmax.

Parameters
----------
cls : float, array or tuple of arrays
  Either one cl (1D array) or a tuple of either 4 cl
  or of n*(n+1)/2 cl.
  Some of the cl may be None, implying no
  cross-correlation. See *new* parameter.
lmax : int, scalar, optional
  The lmax (if None or <0, the largest size-1 of cls)
mmax : int, scalar, optional
  The mmax (if None or <0, =lmax)
new : bool, optional
  If True, use the new ordering of cl's, ie by diagonal
  (e.g. TT, EE, BB, TE, EB, TB or TT, EE, BB, TE if 4 cl as input).
  If False, use the old ordering, ie by row
  (e.g. TT, TE, TB, EE, EB, BB or TT, TE, EE, BB if 4 cl as input).

Returns
-------
alms : array or list of arrays
  the generated alm if one spectrum is given, or a list o

In [ ]:
# Generate sample alm data for testing
lmax = estimator_con.lmax
npol = 3
#nelem = (lmax + 1) * (lmax + 2) // 2   # HEALPix alm element count
nelem = hp.Alm.getsize(lmax)

# Create random alm with correct shape
np.random.seed(42)
#alm_test = np.random.randn(npol, nelem) + 1j * np.random.randn(npol, nelem)
cls = np.zeros((4, lmax + 1))
cls[0] = np.ones(lmax + 1)
cls[1] = np.ones(lmax + 1)
cls[2] = np.ones(lmax + 1)
alm_test = hp.synalm((cls[0], cls[1], cls[2], cls[3]), lmax=lmax)
print(f'{alm_test.shape=}')
print(f'{alm_test.dtype=}')

# Try to call compute_estimate_sst
L_list = np.linspace(0, 100, 101) 
Lmax = 100 

print(alm_test.dtype)
print(L_list.dtype)

estimate, cubic, lin_term, fisher = estimator_con.compute_estimate_sst(
    alm_test,
    L_list,
    Lmax,
    theta_batch=25
)

alm_test.shape=(3, 5151)
alm_test.dtype=dtype('complex128')
complex128
float64
(4, 3, 2, 101)
0.0
0
